# Solutions — Redux Toolkit

One solution per exercise and per mini challenge, in lesson order.
Read these **after** you have tried. A solution you have not attempted teaches nothing.

The measured figures quoted here come from running `@reduxjs/toolkit` 2.12.0 with
`react-redux` 9.3.0 on React 19.3.0.

### LESSON 87 — Exercise

In [ ]:
// L87 solution — denormalising, and counting the copies

const l87sFlat = {
  projects: {
    p1: { id: "p1", name: "Apollo", taskIds: ["t1", "t2"] },
    p2: { id: "p2", name: "Gemini", taskIds: ["t3"] },
  },
  tasks: {
    t1: { id: "t1", title: "Design", assigneeId: "u1", projectId: "p1" },
    t2: { id: "t2", title: "Build", assigneeId: "u2", projectId: "p1" },
    t3: { id: "t3", title: "Plan", assigneeId: "u1", projectId: "p2" },
  },
  users: { u1: { id: "u1", name: "Ada" }, u2: { id: "u2", name: "Grace" } },
};

// 1. rebuild the nested shape for one project
function l87sDenormalise(flat, projectId) {
  const project = flat.projects[projectId];
  return {
    ...project,
    tasks: project.taskIds.map((id) => {
      const task = flat.tasks[id];
      return { ...task, assignee: flat.users[task.assigneeId] };
    }),
  };
}

console.log("denormalised p1:", JSON.stringify(l87sDenormalise(l87sFlat, "p1"), null, 1));

// 2. adding a task, both ways — counting the objects that must be copied
const l87sNested = {
  projects: [
    { id: "p1", name: "Apollo", tasks: [{ id: "t1", title: "Design", assignee: { id: "u1", name: "Ada" } }] },
    { id: "p2", name: "Gemini", tasks: [] },
  ],
};

const l87sAddNested = (data, projectId, task) => ({
  ...data,                                                    // 1 the root
  projects: data.projects.map((p) =>                          // 2 the array
    p.id === projectId ? { ...p, tasks: [...p.tasks, task] } : p,  // 3 the project, 4 its task array
  ),
});

const l87sAddFlat = (flat, projectId, task) => ({
  ...flat,                                                    // 1 the root
  tasks: { ...flat.tasks, [task.id]: task },                  // 2 the tasks lookup
  projects: {                                                 // 3 the projects lookup
    ...flat.projects,
    [projectId]: { ...flat.projects[projectId], taskIds: [...flat.projects[projectId].taskIds, task.id] },
  },
});

const l87sNew = { id: "t9", title: "Ship", assigneeId: "u1", projectId: "p1" };
console.log("\nnested add  — projects[0] task count:", l87sAddNested(l87sNested, "p1", l87sNew).projects[0].tasks.length);
console.log("flat add    — task ids for p1:", l87sAddFlat(l87sFlat, "p1", l87sNew).projects.p1.taskIds);

// Which would I rather write a reducer for? The flat one — and the reason is not the object count
// (they are similar) but that the flat version's updates are LOOKUPS, not searches. Every nested
// update needs a .map over an array to find the thing first, and that .map has to be repeated,
// correctly, in every reducer that touches a project. Nesting one level deeper adds another map.

// 3. Context or a store?
//   theme toggle              -> Context. One value, rarely changes, read everywhere.
//   logged-in user            -> Context. Same shape; a store is fine too if one already exists.
//   cart used by six screens  -> store. Many updates from many places, and it is the app's data.
//   one dropdown's open state -> neither: useState in the dropdown. It is not shared at all.
//   projects with tasks,
//     edited from 3 screens   -> store. This is the case the topic is about.
//   current locale            -> Context. One value, set once, read widely.

### LESSON 87 — Mini challenge

In [ ]:
// L87 solution — the three lookups, and what normalising costs

const l87sTasksWithAssignee = (flat, projectId) =>
  flat.projects[projectId].taskIds.map((id) => ({
    ...flat.tasks[id],
    assigneeName: flat.users[flat.tasks[id].assigneeId].name,
  }));

const l87sTasksForUser = (flat, userId) =>
  Object.values(flat.tasks).filter((task) => task.assigneeId === userId);

const l87sCountsByProject = (flat) =>
  Object.fromEntries(Object.values(flat.projects).map((p) => [p.id, p.taskIds.length]));

console.log("p1 with names:", l87sTasksWithAssignee(l87sFlat, "p1").map((t) => `${t.title}/${t.assigneeName}`));
console.log("u1's tasks   :", l87sTasksForUser(l87sFlat, "u1").map((t) => t.id));
console.log("counts       :", l87sCountsByProject(l87sFlat));

// 1. The harder one is "every task in a project with its assignee's name". Nested, the name is
//    already sitting on the task; flat, you follow two references. That is the real cost of
//    normalising: reads get more work, writes get less. It is the right trade for state that is
//    edited, and the wrong one for data you only ever display.
//
// 2. They are SELECTORS. And the idea is not new — LESSON 29 called it a derived value, and
//    LESSON 21 had you filter and count before rendering. A selector is a derived value that
//    reads from the store instead of from props.
//
// 3. To make "tasks for a user" fast, add a `tasksByUser: { u1: ["t1", "t3"] }` index. What it
//    costs: every reassignment must now update TWO places — the task's assigneeId and both users'
//    lists — and if one update is missed the index silently disagrees with the data. That is the
//    duplication normalising was supposed to remove, reintroduced on purpose for speed. Do it
//    when a measurement says to (topic 23), not before.

### LESSON 88 — Exercise

In [ ]:
// L88 solution — two more cases, and Immer by hand

const l88sInitial = { items: [{ id: "t1", text: "write", done: false }], status: "idle" };

function l88sReducer(state = l88sInitial, action) {
  switch (action.type) {
    case "tasks/added":
      return { ...state, items: [...state.items, action.payload] };
    case "tasks/removed":
      return { ...state, items: state.items.filter((item) => item.id !== action.payload) };
    case "tasks/renamed":
      return {
        ...state,
        items: state.items.map((item) =>
          item.id === action.payload.id ? { ...item, text: action.payload.text } : item,
        ),
      };
    default:
      return state;
  }
}

const l88sRenamed = l88sReducer(l88sInitial, { type: "tasks/renamed", payload: { id: "t1", text: "write tests" } });

console.log("renamed:", l88sRenamed.items[0].text);
console.log("input untouched:      ", l88sInitial.items[0].text === "write");
console.log("new object returned:  ", l88sRenamed !== l88sInitial);
console.log("unhandled -> identical:", l88sReducer(l88sRenamed, { type: "x" }) === l88sRenamed);

// 2. the same thing in the Immer style, plus the machinery Immer replaces
function l88sDraft(state, action) {
  switch (action.type) {
    case "tasks/added":
      state.items.push(action.payload);
      break;
    case "tasks/removed":
      state.items = state.items.filter((item) => item.id !== action.payload);
      break;
    case "tasks/renamed": {
      const task = state.items.find((item) => item.id === action.payload.id);
      if (task) task.text = action.payload.text;
      break;
    }
    default:
      break;
  }
}

function l88sApply(state, action) {
  const copy = structuredClone(state);      // a real Immer uses a proxy; this copies
  l88sDraft(copy, action);
  return copy;
}

const l88sViaDraft = l88sApply(l88sInitial, { type: "tasks/renamed", payload: { id: "t1", text: "write tests" } });
console.log("\nvia draft:", l88sViaDraft.items[0].text, "| input untouched:", l88sInitial.items[0].text === "write");

// Which reads better for `renamed`? The draft version, clearly: `task.text = ...` against a
// find-and-map-and-spread. That gap is the whole reason Redux Toolkit ships Immer — and note that
// l88sApply copies the ENTIRE state on every action, which is why the real Immer uses a proxy that
// only copies the parts you actually touched.

**3 — in your project.** Running the lesson's slice against `@reduxjs/toolkit` 2.12.0 prints:

```text
actions exported:      [ 'added', 'cleared' ]
action creator output: { type: 'tasks/added', payload: { id: 1, text: 'write' } }
before.items.length: 0 | after.items.length: 1
new object? true
unknown action returns identical object: true
```

And mutating the state outside a reducer — `store.getState().tasks.items.push(...)` — throws:

```text
TypeError: Cannot add property 1, object is not extensible
```

because Redux Toolkit freezes the state in development. That error message is worth recognising:
it always means a mutation outside a reducer, usually in a component or a selector.

### LESSON 88 — Mini challenge

In [ ]:
// L88 solution — events versus commands

const l88sActions = [
  ["tasks/setItems", "command", "tasks/loaded or tasks/fetchSucceeded — say WHERE the items came from"],
  ["tasks/added", "event", "already past tense; nothing to change"],
  ["ui/openModal", "command", "ui/modalOpened — but see note 2"],
  ["tasks/fetchSucceeded", "event", "fine; createAsyncThunk generates tasks/fetch/fulfilled for you"],
  ["filters/setFilter", "command", "filters/filterChanged, with the filter in the payload"],
  ["auth/loggedOut", "event", "already past tense"],
  ["tasks/updateTaskText", "command", "tasks/textEdited"],
  ["projects/selected", "event", "already past tense"],
  ["ui/toggleSidebar", "command", "ui/sidebarToggled — but see note 2"],
  ["tasks/removeAllDone", "command", "tasks/completedCleared"],
];

for (const [type, kind, note] of l88sActions) {
  console.log(`${type.padEnd(22)} ${kind.padEnd(8)} ${note}`);
}
console.log(`\n${l88sActions.filter(([, k]) => k === "command").length} of ${l88sActions.length} are commands`);

// 1. What a reducer handling tasks/setItems knows: nothing except "replace the list". It cannot
//    behave differently for a first load, a refresh, a search result or an optimistic update,
//    because all four dispatch the same action with the same shape. And a DevTools log of ten
//    setItems actions tells you the list changed ten times and nothing about WHY — which is the
//    one thing the log existed to give you. Events name a cause; commands name an assignment.
//
// 2. The two that are fine as commands are the UI ones: ui/openModal and ui/toggleSidebar. The
//    reason: for UI state there IS no separate cause to name — the user's intent and the state
//    change are the same event, and "the modal was asked to open" adds nothing over "open the
//    modal". The past-tense rule matters for DOMAIN state, where the same change can arise from
//    several different causes you will want to tell apart later.

### LESSON 89 — Exercise

In [ ]:
// L89 solution — selectors, and writing createSelector

const l89sState = {
  tasks: {
    items: [
      { id: "t1", text: "Design", done: true, projectId: "p1" },
      { id: "t2", text: "Build", done: false, projectId: "p1" },
      { id: "t3", text: "Plan", done: false, projectId: "p2" },
    ],
    status: "success",
  },
  filters: { done: "open", projectId: "p1" },
};

// 1.
const l89sSelectByProject = (state, projectId) =>
  state.tasks.items.filter((task) => task.projectId === projectId);

const l89sSelectCountsByProject = (state) =>
  state.tasks.items.reduce((counts, task) => {
    counts[task.projectId] = (counts[task.projectId] ?? 0) + 1;
    return counts;
  }, {});

console.log("p1 tasks:", l89sSelectByProject(l89sState, "p1").map((t) => t.id));
console.log("counts  :", l89sSelectCountsByProject(l89sState));

// Which is safe in useSelector? NEITHER, as written — both build a new array or object on every
// call, so Object.is fails every time and the component re-renders on every dispatch anywhere in
// the app. The difference is in the fix: the counts object needs memoization, while
// selectByProject can often be avoided entirely by selecting state.tasks.items and filtering in
// the component, where the result is used once and thrown away.

// 2. a memoizer that watches only its inputs — this is createSelector
function l89sCreateSelector(inputs, combine) {
  let lastInputs = null;
  let lastResult;
  return (...args) => {
    const values = inputs.map((input) => input(...args));
    const same = lastInputs && values.every((value, i) => Object.is(value, lastInputs[i]));
    if (!same) {
      lastInputs = values;
      lastResult = combine(...values);
    }
    return lastResult;
  };
}

const l89sSelectFilters = (state) => state.filters;
const l89sSelectItems = (state) => state.tasks.items;

const l89sVisible = l89sCreateSelector([l89sSelectItems, l89sSelectFilters], (items, filters) =>
  items.filter(
    (task) =>
      (filters.done === "all" || (filters.done === "open") === !task.done) &&
      task.projectId === filters.projectId,
  ),
);

console.log("\nvisible:", l89sVisible(l89sState).map((t) => t.id));
console.log("stable across calls:", l89sVisible(l89sState) === l89sVisible(l89sState));

// an unrelated part of the state changes: the whole state object is new, but the inputs are not
const l89sUnrelated = { ...l89sState, ui: { sidebar: true } };
console.log("unrelated change -> recomputed?", l89sVisible(l89sUnrelated) !== l89sVisible(l89sState));

// a relevant change
const l89sChanged = { ...l89sState, filters: { ...l89sState.filters, projectId: "p2" } };
console.log("relevant change  -> recomputed?", l89sVisible(l89sChanged) !== l89sVisible(l89sState));
console.log("and it is right :", l89sVisible(l89sChanged).map((t) => t.id));

**Common mistake:** memoizing against the whole state object, as the lesson's `l89Memoize` does.
Every dispatch anywhere replaces the state object, so the "memoized" selector recomputes on every
action — it caches nothing and adds a comparison. The point of input selectors is that they narrow
what "changed" means.

**3 — in your project.** With the inline filtering selector, a component logs a render for every
dispatch, including actions from a completely unrelated slice. Selecting `state.tasks.items` and
filtering in the component takes it to one render per actual change to the items. That is the
whole exercise: the fix is one line, and the bug is invisible until you count.

### LESSON 89 — Mini challenge

In [ ]:
// L89 solution — reducer, selector, or component?

const l89sPlacement = [
  ["adding a task", "reducer", "it changes the data"],
  ["showing only tasks for the selected project", "selector", "a view of the data; the data is unchanged"],
  ["sorting tasks by due date for display", "selector", "presentation order — another screen may want another order"],
  ["marking every task in a project as done", "reducer", "it changes the data, and it is one atomic action"],
  ["formatting a date as '3 days ago'", "component", "depends on the current time, so it is neither state nor derived-from-state"],
  ["counting how many tasks are still open", "selector", "derived from the data, useful in several places"],
  ["deciding whether 'clear completed' is disabled", "component", "a UI decision about one screen's buttons"],
  ["clamping a title to 80 characters when saved", "reducer", "a rule about what the data may contain — enforce it once, at the write"],
];

for (const [what, where, why] of l89sPlacement) {
  console.log(`${where.padEnd(10)} ${what}\n           ${why}`);
}

// The two that could go either way: SORTING (3) and the DISABLED BUTTON (7).
//
// Sorting could live in a reducer — store the items already sorted — and that is usually wrong,
// because it makes one screen's ordering a property of the data, and the second screen that wants
// a different order has to undo it. The disabled button could be a selector
// (selectHasCompletedTasks), and that is defensible when three screens need the same rule.
//
// The rule: ask "is this a fact about the data, or a decision about this screen?" A fact about the
// data belongs in the reducer (it is true regardless of who looks) or in a selector (it is derived
// and true regardless of who looks). A decision about this screen belongs in the component, and
// putting it in the store means every other screen inherits a decision it did not make.
//
// The date formatting is the interesting one: it is a fact about the data AND about the current
// time, and time is not in the store. Anything whose answer changes without an action belongs
// outside the store.

### LESSON 90 — Exercise

In [ ]:
// L90 solution — two independent async operations

const l90sInitial = { items: [], status: "idle", error: null, saving: "idle", saveError: null };

function l90sReducer(state = l90sInitial, action) {
  switch (action.type) {
    case "tasks/fetch/pending":
      return { ...state, status: "loading", error: null };
    case "tasks/fetch/fulfilled":
      return { ...state, status: action.payload.length === 0 ? "empty" : "success", items: action.payload };
    case "tasks/fetch/rejected":
      return { ...state, status: "error", error: action.payload ?? action.error.message };
    case "tasks/save/pending":
      return { ...state, saving: "saving", saveError: null };
    case "tasks/save/fulfilled":
      return { ...state, saving: "idle", items: [...state.items, action.payload] };
    case "tasks/save/rejected":
      return { ...state, saving: "error", saveError: action.payload ?? action.error.message };
    default:
      return state;
  }
}

// a save that fails, WHILE a successful load is on screen
let l90sState = l90sReducer(undefined, { type: "tasks/fetch/pending" });
l90sState = l90sReducer(l90sState, { type: "tasks/fetch/fulfilled", payload: [{ id: "t1" }] });
l90sState = l90sReducer(l90sState, { type: "tasks/save/pending" });
l90sState = l90sReducer(l90sState, { type: "tasks/save/rejected", payload: "could not save" });

console.log("after a failed save:", JSON.stringify(l90sState));
console.log("the list is still on screen:", l90sState.status === "success", "· items:", l90sState.items.length);

// Why one shared `status` would be a bug: the save failure would set status to "error", and the
// component would replace a perfectly good list of tasks with a full-page error — losing what the
// user was looking at because a different operation failed. Two operations that can be in
// different states at the same time need two fields; one field makes "loaded successfully AND the
// save failed" unrepresentable, which is LESSON 67's argument with the sign flipped: a shape can
// be too small as well as too loose.

// 2. the four states, pinned down
const l90sCases = [
  ["loading", [{ type: "tasks/fetch/pending" }]],
  ["success", [{ type: "tasks/fetch/pending" }, { type: "tasks/fetch/fulfilled", payload: [{ id: "t1" }] }]],
  ["empty", [{ type: "tasks/fetch/pending" }, { type: "tasks/fetch/fulfilled", payload: [] }]],
  ["error", [{ type: "tasks/fetch/pending" }, { type: "tasks/fetch/rejected", payload: "server said no" }]],
];

console.log();
for (const [expected, actions] of l90sCases) {
  const final = actions.reduce((state, action) => l90sReducer(state, action), undefined);
  console.log(`${expected.padEnd(9)} -> ${final.status.padEnd(9)} ${final.status === expected ? "PASS" : "FAIL"}`);
}

// Which state does `if (loading) … else render(items)` get wrong? BOTH error and empty. On an
// error it renders an empty list as though the server had said there is nothing — the user sees a
// blank screen and believes it. On an empty result it renders an empty <ul>, which looks like a
// broken page rather than "no tasks yet". This is LESSON 46's whole argument, arriving through
// Redux instead of through useState.

### LESSON 90 — Mini challenge

In [ ]:
// L90 solution — two requests in flight, and the stale one wins

// 1. the bug
let l90sBuggy = l90sReducer(undefined, { type: "tasks/fetch/pending" });          // click p1
l90sBuggy = l90sReducer(l90sBuggy, { type: "tasks/fetch/pending" });               // click p2
l90sBuggy = l90sReducer(l90sBuggy, { type: "tasks/fetch/fulfilled", payload: [{ id: "t3", projectId: "p2" }] });
l90sBuggy = l90sReducer(l90sBuggy, { type: "tasks/fetch/fulfilled", payload: [{ id: "t1", projectId: "p1" }] }); // p1 arrives LAST

console.log("buggy — selected p2, showing:", l90sBuggy.items.map((t) => t.projectId));

// 2. the fix: a request id, ignoring answers that are no longer current
function l90sSafeReducer(state = { ...l90sInitial, requestId: null }, action) {
  switch (action.type) {
    case "tasks/fetch/pending":
      return { ...state, status: "loading", error: null, requestId: action.meta.requestId };
    case "tasks/fetch/fulfilled":
      if (action.meta.requestId !== state.requestId) return state;      // stale: ignore it
      return { ...state, status: action.payload.length === 0 ? "empty" : "success", items: action.payload };
    case "tasks/fetch/rejected":
      if (action.meta.requestId !== state.requestId) return state;
      return { ...state, status: "error", error: action.payload };
    default:
      return state;
  }
}

let l90sSafe = l90sSafeReducer(undefined, { type: "tasks/fetch/pending", meta: { requestId: "r1" } });
l90sSafe = l90sSafeReducer(l90sSafe, { type: "tasks/fetch/pending", meta: { requestId: "r2" } });
l90sSafe = l90sSafeReducer(l90sSafe, { type: "tasks/fetch/fulfilled", meta: { requestId: "r2" }, payload: [{ id: "t3", projectId: "p2" }] });
l90sSafe = l90sSafeReducer(l90sSafe, { type: "tasks/fetch/fulfilled", meta: { requestId: "r1" }, payload: [{ id: "t1", projectId: "p1" }] });

console.log("fixed — selected p2, showing:", l90sSafe.items.map((t) => t.projectId));
console.log("stale answer ignored:", l90sSafe.items.every((t) => t.projectId === "p2"));

// 3. Ignore, or abort?
//    Ignoring is local and cannot fail: the reducer decides, so it works no matter how the request
//    was made, and a response that arrives anyway is harmless. It costs a wasted response — the
//    bytes were downloaded and thrown away.
//    Aborting (signal from the thunkAPI, passed to fetch) stops the work at the source: less
//    network, less server load, and the request never lands. It costs more moving parts, and it
//    cannot be your only defence, because a request can complete before the abort is processed.
//    In practice: abort for expensive requests, and ALWAYS ignore stale results as well. The
//    guard in the reducer is three lines and it is the one that guarantees correctness.